In [ ]:
import torch
import torch.nn as nn
from torchvision import models

class MorphologyModel(nn.Module):
    # MLP của hình dáng quả trứng
    def __init__(self, input_dim: int = 32, hidden_dims: list = [64, 128], output_dim: int = 256):
        super(MorphologyModel, self).__init__()

        layers = []
        in_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(in_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.3))
            in_dim = h_dim

        layers.append(nn.Linear(in_dim, output_dim))
        layers.append(nn.ReLU())

        self.mlp = nn.Sequential(*layers)
        self.output_dim = output_dim

    def forward(self, x):
        return self.mlp(x)


class ImageModel(nn.Module):
    def __init__(self, model_name: str = "resnet50", output_dim: int = 256, pretrained: bool = True):
        super(ImageModel, self).__init__()

        weights = "DEFAULT" if pretrained else None
        backbone = getattr(models, model_name)(weights=weights)

        if model_name.startswith("resnet"):
            in_features = backbone.fc.in_features
            backbone.fc = nn.Sequential(
                nn.Linear(in_features, output_dim),
                nn.ReLU()
            )

        elif model_name.startswith("efficientnet"):
            in_features = backbone.classifier[-1].in_features
            backbone.classifier[-1] = nn.Sequential(
                nn.Linear(in_features, output_dim),
                nn.ReLU()
            )

        # ==========================================
        # THÊM MỚI: HỖ TRỢ VGG16_BN
        # ==========================================
        elif model_name.startswith("vgg"):
            # Lớp cuối cùng của VGG nằm ở classifier[6]
            in_features = backbone.classifier[6].in_features
            backbone.classifier[6] = nn.Sequential(
                nn.Linear(in_features, output_dim),
                nn.ReLU()
            )
        else:
            raise ValueError(f"Model {model_name} is not supported yet.")

        self.backbone = backbone
        self.output_dim = output_dim

    def forward(self, x):
        return self.backbone(x)

class FusionModel(nn.Module):
    # Fusion model.
    def __init__(
        self,
        image_model_name: str = "resnet50",
        image_output_dim: int = 256,
        mlp_input_dim: int = 32,
        mlp_hidden_dims: list = [64, 128],
        mlp_output_dim: int = 256,
        num_classes: int = 2,
        pretrained: bool = True,
    ):
        super(FusionModel, self).__init__()

        self.image_model = ImageModel(
            model_name=image_model_name,
            output_dim=image_output_dim,
            pretrained=pretrained
        )
        self.morphology_model = MorphologyModel(
            input_dim=mlp_input_dim,
            hidden_dims=mlp_hidden_dims,
            output_dim=mlp_output_dim
        )

        fused_dim = image_output_dim + mlp_output_dim
        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes),
            # nn.Softmax(dim=1)
        )

    def forward(self, image, vector):
        #    image:  Tensor [Batch_size, C, H, W] - ảnh đầu vào
        #    vector: Tensor [Batch_size, số lượng thông tin của các feature hình học]
        #   out: Tensor [Batch_size, 2] - xác suất 2 class

        img_feat = self.image_model(image)
        vec_feat = self.morphology_model(vector)
        fused = torch.cat([img_feat, vec_feat], dim=1)
        out = self.classifier(fused)
        return out


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch

# 1. KHỞI TẠO MÔ HÌNH VỚI THÔNG SỐ THỰC TẾ
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"[*] Đang sử dụng thiết bị: {device}")

# Đếm chính xác số lượng cột số (features) trong file CSV của bạn để điền vào mlp_input_dim
SO_LUONG_FEATURE_CSV = 14

fusion_model = FusionModel(
    image_model_name="vgg16_bn",
    image_output_dim=256,
    mlp_input_dim=SO_LUONG_FEATURE_CSV, # <- Truyền số lượng cột CSV vào đây
    mlp_hidden_dims=[64, 128],
    mlp_output_dim=256,
    num_classes=2,
    pretrained=True # Để True nếu bạn muốn lấy pretrained của ImageNet ban đầu, hoặc False nếu bạn tự load file .pth sau
).to(device)

print("[+] Đã khởi tạo thành công FusionModel!")

# ==========================================
# 2. KIỂM TRA LUỒNG DỮ LIỆU (SANITY CHECK)
# ==========================================
print("-" * 40)
print("🚀 Đang kiểm tra luồng dữ liệu (Forward Pass)...")

BATCH_SIZE = 8

# Giả lập 8 bức ảnh RGB (Batch_size=8, Channels=3, H=224, W=224)
dummy_images = torch.randn(BATCH_SIZE, 3, 224, 224).to(device)

# Giả lập 8 vector hình học từ CSV (Batch_size=8, Features=14)
dummy_vectors = torch.randn(BATCH_SIZE, SO_LUONG_FEATURE_CSV).to(device)

# Đưa qua mô hình
fusion_model.eval()
with torch.no_grad():
    predictions = fusion_model(dummy_images, dummy_vectors)

# Kiểm tra kết quả
print(f"[*] Kích thước tensor Hình ảnh đầu vào: {dummy_images.shape}")
print(f"[*] Kích thước tensor CSV đầu vào: {dummy_vectors.shape}")
print(f"[*] Kích thước Đầu ra (Logits): {predictions.shape}")
# Mong đợi: [8, 2] -> 8 quả trứng, mỗi quả có 2 điểm số (Trống/Mái)

if predictions.shape == (BATCH_SIZE, 2):
    print("🎉 CHÚC MỪNG! Dữ liệu đã đi qua hệ thống Fusion một cách hoàn hảo!")
else:
    print("⚠️ Có lỗi về kích thước Tensor.")

[*] Đang sử dụng thiết bị: cpu
Downloading: "https://download.pytorch.org/models/vgg16_bn-6c64b313.pth" to /root/.cache/torch/hub/checkpoints/vgg16_bn-6c64b313.pth


100%|██████████| 528M/528M [00:08<00:00, 63.7MB/s]


[+] Đã khởi tạo thành công FusionModel!
----------------------------------------
🚀 Đang kiểm tra luồng dữ liệu (Forward Pass)...
[*] Kích thước tensor Hình ảnh đầu vào: torch.Size([8, 3, 224, 224])
[*] Kích thước tensor CSV đầu vào: torch.Size([8, 14])
[*] Kích thước Đầu ra (Logits): torch.Size([8, 2])
🎉 CHÚC MỪNG! Dữ liệu đã đi qua hệ thống Fusion một cách hoàn hảo!


In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from PIL import Image
from tqdm.auto import tqdm

# --- 1. FILTER DATA FROM CSV ---
# UPDATE THESE PATHS TO YOUR GOOGLE DRIVE LOCATIONS
CSV_PATH = r"/content/drive/MyDrive/Major_Project/linked_egg_metadata_qa_checked.csv"
IMAGE_DIR = r"/content/drive/MyDrive/Major_Project/merged_dataset_rgb"

# Load the metadata
df = pd.read_csv(CSV_PATH)

# Strictly filter out images that failed the QA check (if the column exists)
if 'QA_Failed' in df.columns:
    clean_df = df[df['QA_Failed'] == False].copy()
else:
    clean_df = df.copy()

# Define a dictionary to map string labels to integer classes for PyTorch
# 'T' (Trống / Infertile) -> 0
# 'M' (Mái / Fertile) -> 1
LABEL_MAP = {'T': 0, 'M': 1}

# Create the data list format: [(img_path, label), ...]
data_list = []

# Create a separate list just for the labels (needed for Stratified K-Fold splitting)
all_labels = []

missing_files = 0

for _, row in clean_df.iterrows():
    # Use os.path.basename to extract just the filename (e.g., "egg_1.jpg")
    # in case the 'image_path' column contains full outdated paths.
    filename = os.path.basename(str(row['image_path']))
    img_path = os.path.join(IMAGE_DIR, filename)

    # Extract the label, strip any accidental whitespace, and convert to uppercase
    str_label = str(row['label']).strip().upper()

    # Check if the label is valid ('T' or 'M') and if the image file actually exists
    if str_label in LABEL_MAP:
        if os.path.exists(img_path):
            int_label = LABEL_MAP[str_label]

            # Append to both lists
            data_list.append((img_path, int_label))
            all_labels.append(int_label)
        else:
            missing_files += 1

print(f"[*] Successfully loaded {len(data_list)} clean images for training.")
print(f"[*] Extracted {len(all_labels)} labels for K-Fold stratification.")

if missing_files > 0:
    print(f"⚠️ Warning: {missing_files} images were listed in the CSV but not found in the directory.")

[*] Successfully loaded 4616 clean images for training.
[*] Extracted 4616 labels for K-Fold stratification.
⚠️ Warning: 2 images were listed in the CSV but not found in the directory.


In [ ]:
import os
import gc
import copy
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from tqdm.auto import tqdm
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ==========================================
# 0. CẤU HÌNH & THIẾT BỊ
# ==========================================
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"[*] Đang sử dụng thiết bị: {device}")

SAVE_DIR = r"/content/drive/MyDrive/Major_Project/checkpoints_fusion"
os.makedirs(SAVE_DIR, exist_ok=True)
BATCH_SIZE = 16
NUM_EPOCHS = 40

print("="*50)
print("BẮT ĐẦU HUẤN LUYỆN FUSION MODEL (IMAGE + CSV)")
print("="*50)

# ==========================================
# 1. ĐỊNH NGHĨA MODEL & DATASET
# ==========================================
# (Chèn 3 class MorphologyModel, ImageModel, FusionModel của bạn vào đây)
class MorphologyModel(nn.Module):
    def __init__(self, input_dim: int = 32, hidden_dims: list = [64, 128], output_dim: int = 256):
        super(MorphologyModel, self).__init__()
        layers = []
        in_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(in_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.3))
            in_dim = h_dim
        layers.append(nn.Linear(in_dim, output_dim))
        layers.append(nn.ReLU())
        self.mlp = nn.Sequential(*layers)
        self.output_dim = output_dim

    def forward(self, x):
        return self.mlp(x)

class ImageModel(nn.Module):
    def __init__(self, model_name: str = "vgg16_bn", output_dim: int = 256, pretrained: bool = True):
        super(ImageModel, self).__init__()
        weights = "DEFAULT" if pretrained else None
        backbone = getattr(models, model_name)(weights=weights)

        if model_name.startswith("vgg"):
            in_features = backbone.classifier[6].in_features
            backbone.classifier[6] = nn.Sequential(
                nn.Linear(in_features, output_dim),
                nn.ReLU()
            )
        self.backbone = backbone
        self.output_dim = output_dim

    def forward(self, x):
        return self.backbone(x)

class FusionModel(nn.Module):
    def __init__(self, image_model_name: str = "vgg16_bn", image_output_dim: int = 256,
                 mlp_input_dim: int = 32, mlp_hidden_dims: list = [64, 128],
                 mlp_output_dim: int = 256, num_classes: int = 2, pretrained: bool = True):
        super(FusionModel, self).__init__()
        self.image_model = ImageModel(model_name=image_model_name, output_dim=image_output_dim, pretrained=pretrained)
        self.morphology_model = MorphologyModel(input_dim=mlp_input_dim, hidden_dims=mlp_hidden_dims, output_dim=mlp_output_dim)

        fused_dim = image_output_dim + mlp_output_dim
        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )

    def forward(self, image, vector):
        img_feat = self.image_model(image)
        vec_feat = self.morphology_model(vector)
        fused = torch.cat([img_feat, vec_feat], dim=1)
        return self.classifier(fused)

class FusionEggDataset(Dataset):
    def __init__(self, data_list, morph_features, transform=None):
        self.data_list = data_list
        self.morph_features = torch.tensor(morph_features, dtype=torch.float32)
        self.transform = transform

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        img_path, label = self.data_list[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform: image = self.transform(image)
        vector = self.morph_features[idx]
        return image, vector, label





[*] Đang sử dụng thiết bị: cpu
BẮT ĐẦU HUẤN LUYỆN FUSION MODEL (IMAGE + CSV)


In [ ]:
from sklearn.impute import SimpleImputer
# ==========================================
# 2. DATA PROCESSING & SYNCHRONIZATION (UPGRADED VERSION)
# ==========================================
print("\n🔄 Synchronizing Image and CSV data...")

# STEP 1: Convert the image list (combined_data) into a Pandas DataFrame
df_images = pd.DataFrame(data_list, columns=['full_image_path', 'label'])

# Extract the 'filename' (e.g., 'egg_1.jpg') to act as the synchronization key
df_images['filename'] = df_images['full_image_path'].apply(lambda x: os.path.basename(str(x)))

# STEP 2: Read the Morphology CSV file
CSV_MORPH_PATH = r"/content/drive/MyDrive/Major_Project/all_features_label_2.csv"
df_morph = pd.read_csv(CSV_MORPH_PATH)

# ⚠️ IMPORTANT: Find the synchronization key in the CSV file.
# Look at your CSV file and find the column that contains the image names.
# Replace 'image_path' below with the exact name of that column in your CSV.
CSV_FILENAME_COLUMN = 'image_path'

if CSV_FILENAME_COLUMN in df_morph.columns:
    df_morph['filename'] = df_morph[CSV_FILENAME_COLUMN].apply(lambda x: os.path.basename(str(x)))
else:
    raise ValueError(f"❌ Critical Error: The CSV file does not contain the column '{CSV_FILENAME_COLUMN}'. Please check your CSV to find the correct column name containing the image filenames!")

# STEP 3: Merge the two tables (Inner Join) - Keep only the eggs present in BOTH datasets
df_merged = pd.merge(df_images, df_morph, on='filename', how='inner')
print(f"[*] Synchronization successful! Total valid samples retained: {len(df_merged)}")

# ==========================================
# STEP 4: FEATURE SELECTION & CLEANING
# ==========================================
# Retrieve the synchronized image paths and labels
synced_image_paths = df_merged['full_image_path'].tolist()
synced_labels = df_merged['label_x'].tolist() if 'label_x' in df_merged.columns else df_merged['label'].tolist()

# 1. Automatically select ONLY columns that contain numbers
numeric_df = df_merged.select_dtypes(include=['number', 'float', 'int'])

# --- THE FIX: STRICT FEATURE SELECTION ---
# We explicitly define all metadata, IDs, and labels that MUST be removed
# to prevent noise and data leakage.
columns_to_drop = [
    # Metadata & IDs (Noise)
    'O', 'egg_id_x', 'egg_id_y', 'batch', 'day', 'ngay', 'dot',
    'side_x', 'side_y', 'img_key',

    # Path strings (If they accidentally got parsed as numbers/categories)
    'img_path', 'image_path', 'full_image_path', 'filename',

    # Target Labels & Status (Data Leakage)
    'label', 'label_x', 'label_y', 'Label', 'QA_Failed',

    # Length, Width (Not sufficient)
    'length_mm', 'width_mm'

]

# Drop these columns if they exist in the dataframe
numeric_df = numeric_df.drop(columns=[col for col in columns_to_drop if col in numeric_df.columns])
# --- DISPLAY SELECTED FEATURES ---
# Get the list of remaining columns and print them
selected_features = numeric_df.columns.tolist()
print(f"[*] Successfully selected {len(selected_features)} geometric/color features:")
print(f"    -> {selected_features}")

# 2. Handle Infinity and NaNs safely
import numpy as np
numeric_df = numeric_df.replace([np.inf, -np.inf], np.nan)
numeric_df = numeric_df.fillna(0)

# 3. Convert the perfectly clean dataframe to a numpy array
raw_morph_features = numeric_df.values

# Safety checks
assert not np.isnan(raw_morph_features).any(), "❌ Critical Error: NaNs are still present!"
assert not np.isinf(raw_morph_features).any(), "❌ Critical Error: Infinities are still present!"

mlp_input_dim = raw_morph_features.shape[1]
print(f"[*] MLP Input Dimension optimally reduced to: {mlp_input_dim} geometric/color features.")

# ==========================================
# TRAIN / VAL SPLIT ON SYNCHRONIZED DATA
# ==========================================
# ... (Keep your train_test_split and StandardScaler code the same as before) ...

# ==========================================
# TRAIN / VAL SPLIT ON SYNCHRONIZED DATA
# ==========================================
indices = np.arange(len(synced_image_paths))

train_idx, val_idx = train_test_split(indices, test_size=0.2, stratify=synced_labels, random_state=42)

# Extract Train / Val data based on synchronized indices
train_data_list = [(synced_image_paths[i], synced_labels[i]) for i in train_idx]
val_data_list = [(synced_image_paths[i], synced_labels[i]) for i in val_idx]
train_labels = [synced_labels[i] for i in train_idx]

# --- NUMERICAL DATA SCALING ---
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# Now this will work perfectly because 'numeric_df' only contains numbers!
train_scaled_morph = scaler.fit_transform(raw_morph_features[train_idx])
val_scaled_morph = scaler.transform(raw_morph_features[val_idx])

# -> NOW YOU CAN CONTINUE WITH PART 3 (DATALOADER) AS BEFORE


🔄 Synchronizing Image and CSV data...
[*] Synchronization successful! Total valid samples retained: 4573
[*] Successfully selected 27 geometric/color features:
    -> ['Lx', 'Ly', 'Lxt', 'Lxh', 'Wh50', 'Wh85', 'Wh90', 'Wt50', 'Wt85', 'Wt90', 'Sx', 'Sxt', 'Sxh', 'Sxd', 'Seh', 'Set', 'No303', 'YR90', 'R85', 'L85', 'GPT', 'GYR90', 'Gr', 'R50BX', 'DFH', 'DSY', 'Gym']
[*] MLP Input Dimension optimally reduced to: 27 geometric/color features.


In [ ]:
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import DataLoader
# from torchvision import transforms
# from tqdm.auto import tqdm
# import copy
# import os
# from collections import Counter

# # ==========================================
# # 3. DATALOADER & TRANSFORMS (UPDATED)
# # ==========================================
# # Cấu hình kích thước Batch Size (Bạn có thể tăng lên 32 nếu GPU có đủ RAM)
# BATCH_SIZE = 16

# # Transform cho tập Train (Có Data Augmentation để giảm Overfitting)
# train_transforms = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomRotation(15),
#     transforms.ToTensor(),
#     transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
# ])

# # Transform cho tập Validation (Không có Augmentation, chỉ chuẩn hóa)
# val_transforms = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
# ])

# # Khởi tạo các Custom Dataset
# # Chú ý: 'train_data_list', 'train_scaled_morph', 'train_labels' đã được tạo ở bước Đồng bộ trước đó.
# train_dataset = FusionEggDataset(
#     data_list=train_data_list,
#     morph_features=train_scaled_morph,
#     transform=train_transforms
# )

# val_dataset = FusionEggDataset(
#     data_list=val_data_list,
#     morph_features=val_scaled_morph,
#     transform=val_transforms
# )

# # Tạo DataLoader
# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
# val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
# # ==========================================
# # 4. INITIALIZE MODEL & TRAINING SETUP
# # ==========================================
# device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# print(f"[*] Using device: {device}")

# # 1. Initialize the Fusion Model with dynamic mlp_input_dim
# model = FusionModel(
#     image_model_name="vgg16_bn",
#     mlp_input_dim=mlp_input_dim, # Automatically uses 35 features
#     pretrained=False # Set to False because we are loading your custom weights!
# ).to(device)

# # ---------------------------------------------------------
# # 2. LOAD PRETRAINED CUSTOM WEIGHTS (.pth)
# # ---------------------------------------------------------
# weights_path = "/content/drive/MyDrive/Major_Project/checkpoints_bv/vgg16_bn_phase2_bv_fold_2.pth"

# print(f"\n🔄 Loading pre-trained weights from: {weights_path}")
# try:
#     # Load the state dictionary from your saved file
#     state_dict = torch.load(weights_path, map_location=device)

#     # Filter out the final classification layer ('classifier.6')
#     # because our FusionModel uses a 256-dimensional output there instead of 2.
#     filtered_state_dict = {k: v for k, v in state_dict.items() if not k.startswith('classifier.6')}

#     # Inject the weights into the VGG backbone of the FusionModel
#     model.image_model.backbone.load_state_dict(filtered_state_dict, strict=False)
#     print("[+] Successfully loaded custom blood vessel weights into the FusionModel!")

# except FileNotFoundError:
#     print(f"❌ Error: Could not find the file at {weights_path}. Please check your Google Drive mount.")
# except Exception as e:
#     print(f"❌ An error occurred while loading weights: {e}")

# # ---------------------------------------------------------
# # 3. TRAINING PARAMETERS (THE "FREEZE" STRATEGY)
# # ---------------------------------------------------------
# print("\n❄️ Freezing the VGG backbone to protect pre-trained weights...")

# # 1. FREEZE THE VGG BACKBONE
# for param in model.image_model.backbone.parameters():
#     param.requires_grad = False

# # Ensure MLP and Classifier are UNfrozen (they need to learn)
# for param in model.morphology_model.parameters():
#     param.requires_grad = True
# for param in model.classifier.parameters():
#     param.requires_grad = True

# # Calculate Class Weights to handle class imbalance
# label_counts = Counter(train_labels)
# weight_0 = len(train_labels) / (2.0 * label_counts.get(0, 1))
# weight_1 = len(train_labels) / (2.0 * label_counts.get(1, 1))
# class_weights = torch.tensor([weight_0, weight_1], dtype=torch.float).to(device)

# criterion = nn.CrossEntropyLoss(weight=class_weights)

# # 2. OPTIMIZER: Only pass parameters that REQUIRE gradients
# # We use a slightly higher Learning Rate (5e-4) here because we are only
# # training the newly initialized linear layers, which need to learn fast.
# trainable_params = filter(lambda p: p.requires_grad, model.parameters())
# optimizer = optim.AdamW(trainable_params, lr=5e-4, weight_decay=1e-4)

# # 3. SCHEDULER
# scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

# print("[+] Optimizer setup complete. Only MLP and Classifier will be updated.")

# # ==========================================
# # 5. THE TRAINING LOOP (ENHANCED TRACKING)
# # ==========================================
# NUM_EPOCHS = 40
# SAVE_DIR = r"/content/drive/MyDrive/Major_Project/checkpoints_fusion"
# os.makedirs(SAVE_DIR, exist_ok=True)

# best_acc = 0.0
# best_model_wts = copy.deepcopy(model.state_dict())
# epochs_no_improve = 0
# patience_limit = 99

# print("\n" + "="*50)
# print(f"🚀 STARTING FUSION MODEL TRAINING (Batch Size: {BATCH_SIZE})")
# print("="*50)

# for epoch in range(NUM_EPOCHS):
#     # ---------- TRAINING PHASE ----------
#     model.train()
#     running_loss, running_corrects = 0.0, 0
#     processed_samples = 0  # To keep track of samples seen so far in this epoch

#     # Get the current learning rate from the optimizer
#     current_lr = optimizer.param_groups[0]['lr']

#     train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]", leave=False)

#     for images, vectors, labels in train_pbar:
#         images, vectors, labels = images.to(device), vectors.to(device), labels.to(device)

#         optimizer.zero_grad()
#         outputs = model(image=images, vector=vectors)

#         _, preds = torch.max(outputs, 1)

#         loss = criterion(outputs, labels)
#         loss.backward()

#         # Gradient clipping for stability
#         torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
#         optimizer.step()

#         # Accumulate metrics
#         batch_size = images.size(0)
#         running_loss += loss.item() * batch_size
#         running_corrects += torch.sum(preds == labels.data)
#         processed_samples += batch_size

#         # Calculate real-time running averages
#         running_avg_loss = running_loss / processed_samples
#         running_acc = (running_corrects.double() / processed_samples).item()

#         # Update the progress bar with detailed metrics
#         train_pbar.set_postfix({
#             'loss': f"{loss.item():.4f}",
#             'avg_loss': f"{running_avg_loss:.4f}",
#             'acc': f"{running_acc:.4f}",
#             'lr': f"{current_lr:.1e}"
#         })

#     # Calculate final epoch metrics
#     train_loss = running_loss / len(train_dataset)
#     train_acc = running_corrects.double() / len(train_dataset)

#     # ---------- VALIDATION PHASE ----------
#     model.eval()
#     val_loss, val_corrects = 0.0, 0
#     val_processed_samples = 0

#     val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]", leave=False)

#     with torch.no_grad():
#         for images, vectors, labels in val_pbar:
#             images, vectors, labels = images.to(device), vectors.to(device), labels.to(device)

#             outputs = model(image=images, vector=vectors)
#             _, preds = torch.max(outputs, 1)
#             loss = criterion(outputs, labels)

#             # Accumulate metrics
#             batch_size = images.size(0)
#             val_loss += loss.item() * batch_size
#             val_corrects += torch.sum(preds == labels.data)
#             val_processed_samples += batch_size

#             # Calculate real-time running averages for Validation
#             val_avg_loss = val_loss / val_processed_samples
#             val_running_acc = (val_corrects.double() / val_processed_samples).item()

#             val_pbar.set_postfix({
#                 'loss': f"{loss.item():.4f}",
#                 'avg_loss': f"{val_avg_loss:.4f}",
#                 'acc': f"{val_running_acc:.4f}"
#             })

#     val_loss = val_loss / len(val_dataset)
#     val_acc = val_corrects.double() / len(val_dataset)

#     # Step the scheduler based on validation loss
#     scheduler.step(val_loss)

#     print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

#     # ---------- EARLY STOPPING & MODEL SAVING ----------
#     if val_acc > best_acc:
#         best_acc = val_acc
#         best_model_wts = copy.deepcopy(model.state_dict())
#         epochs_no_improve = 0
#     else:
#         epochs_no_improve += 1

#     if epochs_no_improve >= patience_limit:
#         print(f"[*] Early stopping activated at Epoch {epoch + 1}!")
#         break

# # ---------- SAVE BEST MODEL ----------
# model_path = os.path.join(SAVE_DIR, "best_fusion_model.pth")
# torch.save(best_model_wts, model_path)
# print("\n" + "="*50)
# print(f"🎉 TRAINING COMPLETE. Best Validation Accuracy: {best_acc:.4f}")
# print("="*50)

In [ ]:
import os
from torch.utils.data import DataLoader, random_split

# 1. Define the path to the filtered blood vessel dataset
bv_dir = r"/content/drive/MyDrive/Major_Project/filtered_dataset_blood_vessel_ori"

# 2. Extract image paths and labels into a list (similar to 'clean_data')
bv_data = []
class_mapping = {'0_Male': 0, '1_Female': 1}

print("SCANNING BLOOD VESSEL DIRECTORY...\n" + "-"*40)
for class_folder, label_idx in class_mapping.items():
    folder_path = os.path.join(bv_dir, class_folder)

    if not os.path.exists(folder_path):
        print(f"Warning: {folder_path} does not exist.")
        continue

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            img_path = os.path.join(folder_path, filename)
            bv_data.append((img_path, label_idx))

print(f"Total blood vessel images found: {len(bv_data)}")

SCANNING BLOOD VESSEL DIRECTORY...
----------------------------------------
Total blood vessel images found: 186


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from collections import Counter
from tqdm.auto import tqdm
import copy

# ==========================================
# 1. DATA SYNCHRONIZATION FOR BLOOD VESSEL SUBSET
# ==========================================
print("\n🔄 Synchronizing 186 Blood Vessel images with CSV Data...")

# Convert the specific bv_data list to a DataFrame
df_images = pd.DataFrame(bv_data, columns=['full_image_path', 'label'])
df_images['filename'] = df_images['full_image_path'].apply(lambda x: os.path.basename(str(x)))

# Read the full Morphology CSV file
CSV_MORPH_PATH = r"/content/drive/MyDrive/Major_Project/all_features_label_2.csv"
df_morph = pd.read_csv(CSV_MORPH_PATH)

# Assume the CSV image column is 'image_path'. Update if necessary.
CSV_FILENAME_COLUMN = 'image_path'
df_morph['filename'] = df_morph[CSV_FILENAME_COLUMN].apply(lambda x: os.path.basename(str(x)))

# INNER JOIN: This will automatically filter the CSV down to ONLY our 186 eggs!
df_merged = pd.merge(df_images, df_morph, on='filename', how='inner')
print(f"[*] Synchronization successful! Retained samples: {len(df_merged)} (out of {len(bv_data)})")

# ==========================================
# 2. FEATURE SELECTION & CLEANING
# ==========================================
synced_image_paths = df_merged['full_image_path'].tolist()
synced_labels = df_merged['label_x'].tolist() if 'label_x' in df_merged.columns else df_merged['label'].tolist()

numeric_df = df_merged.select_dtypes(include=['number', 'float', 'int'])

# Strictly drop metadata to prevent data leakage
columns_to_drop = ['O', 'egg_id_x', 'egg_id_y', 'batch', 'day', 'ngay', 'dot',
                   'side_x', 'side_y', 'img_key', 'img_path', 'image_path',
                   'full_image_path', 'filename', 'label', 'label_x', 'label_y',
                   'Label', 'QA_Failed', 'length_mm', 'width_mm']

numeric_df = numeric_df.drop(columns=[col for col in columns_to_drop if col in numeric_df.columns])

# Handle NaNs and Infs safely
numeric_df = numeric_df.replace([np.inf, -np.inf], np.nan)
numeric_df = numeric_df.fillna(0)

raw_morph_features = numeric_df.values
mlp_input_dim = raw_morph_features.shape[1]
print(f"[*] Input Dimension for MLP: {mlp_input_dim} features.")




🔄 Synchronizing 186 Blood Vessel images with CSV Data...
[*] Synchronization successful! Retained samples: 0 (out of 186)
[*] Input Dimension for MLP: 27 features.


In [ ]:
import os
import re
import pandas as pd

# ==========================================
# 1. DATA SYNCHRONIZATION VIA EGG ID
# ==========================================
print("\n🔄 Synchronizing 186 Blood Vessel images with CSV Data using Egg ID...")

df_images = pd.DataFrame(bv_data, columns=['full_image_path', 'label'])
df_images['filename'] = df_images['full_image_path'].apply(lambda x: os.path.basename(str(x)).strip().lower())

# --- STEP A: EXTRACT ID FROM FOLDER IMAGES ---
# Example input: '0_144.jpg' -> Output: '144'
def get_folder_id(filename):
    match = re.search(r'_(\d+)\.jpg', filename)
    return match.group(1) if match else None

df_images['egg_id'] = df_images['filename'].apply(get_folder_id)

# --- STEP B: READ CSV AND EXTRACT ID ---
CSV_MORPH_PATH = r"/content/drive/MyDrive/Major_Project/all_features_label_2.csv"
df_morph = pd.read_csv(CSV_MORPH_PATH)

# Note: Using 'image_path' based on your previous logs
CSV_FILENAME_COLUMN = 'image_path'
df_morph['filename'] = df_morph[CSV_FILENAME_COLUMN].apply(lambda x: str(x).strip().lower())

# Example input: 'dot_1_ngay_0_egg_101_side_a.jpg' -> Output: '101'
def get_csv_id(filename):
    match = re.search(r'egg_(\d+)_side', filename)
    return match.group(1) if match else None

df_morph['egg_id'] = df_morph['filename'].apply(get_csv_id)

# --- STEP C: HANDLE MULTIPLE SIDES (side_a, side_b) ---
# Since the CSV contains both side_a and side_b for a single egg, merging directly
# by egg_id will cause duplicates. We only keep the first occurrence to be safe.
df_morph_unique = df_morph.drop_duplicates(subset=['egg_id'], keep='first')

# --- DEBUG: CHECK EXTRACTED IDs ---
print("\n🔍 DEBUG INFO (Extracted IDs):")
print("Top 5 Folder IDs :", df_images['egg_id'].head(5).tolist())
print("Top 5 CSV IDs    :", df_morph_unique['egg_id'].head(5).tolist())
print("-" * 50)

# --- STEP D: INNER JOIN BY EGG ID ---
df_merged = pd.merge(df_images, df_morph_unique, on='egg_id', how='inner')
print(f"[*] Synchronization successful! Retained samples: {len(df_merged)} (out of {len(bv_data)})")


🔄 Synchronizing 186 Blood Vessel images with CSV Data using Egg ID...

🔍 DEBUG INFO (Extracted IDs):
Top 5 Folder IDs : ['144', '145', '157', '176', '158']
Top 5 CSV IDs    : ['101', '102', '105', '107', '108']
--------------------------------------------------
[*] Synchronization successful! Retained samples: 186 (out of 186)


In [ ]:
# import numpy as np
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import DataLoader
# from torchvision import transforms
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler
# from collections import Counter
# from tqdm.auto import tqdm
# import copy
# import os

# # ==========================================
# # 2. FEATURE SELECTION & CLEANING
# # ==========================================
# print("\n🧹 Cleaning data and extracting features...")

# # Retrieve the synchronized image paths and labels
# synced_image_paths = df_merged['full_image_path'].tolist()
# synced_labels = df_merged['label_x'].tolist() if 'label_x' in df_merged.columns else df_merged['label'].tolist()

# numeric_df = df_merged.select_dtypes(include=['number', 'float', 'int'])

# # Strictly drop metadata to prevent data leakage (including the new 'egg_id')
# columns_to_drop = ['O', 'egg_id', 'egg_id_x', 'egg_id_y', 'batch', 'day', 'ngay', 'dot',
#                    'side_x', 'side_y', 'img_key', 'img_path', 'image_path',
#                    'full_image_path', 'filename', 'label', 'label_x', 'label_y',
#                    'Label', 'QA_Failed', 'length_mm', 'width_mm']

# numeric_df = numeric_df.drop(columns=[col for col in columns_to_drop if col in numeric_df.columns])

# # Handle NaNs and Infs safely
# numeric_df = numeric_df.replace([np.inf, -np.inf], np.nan)
# numeric_df = numeric_df.fillna(0)

# raw_morph_features = numeric_df.values
# mlp_input_dim = raw_morph_features.shape[1]
# print(f"[*] MLP Input Dimension optimally reduced to: {mlp_input_dim} features.")

# # ==========================================
# # 3. SPLIT, SCALE, AND DATALOADERS
# # ==========================================
# # Split the small dataset (80% Train, 20% Val)
# indices = np.arange(len(synced_image_paths))
# train_idx, val_idx = train_test_split(indices, test_size=0.2, stratify=synced_labels, random_state=42)

# train_data_list = [(synced_image_paths[i], synced_labels[i]) for i in train_idx]
# val_data_list = [(synced_image_paths[i], synced_labels[i]) for i in val_idx]
# train_labels = [synced_labels[i] for i in train_idx]

# # Scale features based ONLY on training set
# scaler = StandardScaler()
# train_scaled_morph = scaler.fit_transform(raw_morph_features[train_idx])
# val_scaled_morph = scaler.transform(raw_morph_features[val_idx])

# # Using Batch Size 8 since the dataset is very small
# BATCH_SIZE = 8

# train_transforms = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomRotation(15),
#     transforms.ToTensor(),
#     transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
# ])

# val_transforms = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
# ])

# # Assuming FusionEggDataset is already defined in your environment
# train_dataset = FusionEggDataset(train_data_list, train_scaled_morph, train_transforms)
# val_dataset = FusionEggDataset(val_data_list, val_scaled_morph, val_transforms)

# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
# val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# # ==========================================
# # 4. INITIALIZE MODEL & SETUP DIFFERENTIAL LR
# # ==========================================
# device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# model = FusionModel(
#     image_model_name="vgg16_bn",
#     mlp_input_dim=mlp_input_dim,
#     pretrained=False
# ).to(device)

# # Load the custom blood vessel weights into the VGG backbone
# weights_path = "/content/drive/MyDrive/Major_Project/checkpoints_bv/vgg16_bn_phase2_bv_fold_2.pth"
# if os.path.exists(weights_path):
#     state_dict = torch.load(weights_path, map_location=device)
#     filtered_state_dict = {k: v for k, v in state_dict.items() if not k.startswith('classifier.6')}
#     model.image_model.backbone.load_state_dict(filtered_state_dict, strict=False)
#     print(f"[+] Loaded custom VGG weights from {weights_path}")

# # Calculate Class Weights
# label_counts = Counter(train_labels)
# weight_0 = len(train_labels) / (2.0 * label_counts.get(0, 1))
# weight_1 = len(train_labels) / (2.0 * label_counts.get(1, 1))
# class_weights = torch.tensor([weight_0, weight_1], dtype=torch.float).to(device)
# criterion = nn.CrossEntropyLoss(weight=class_weights)

# # Differential Learning Rates (Unfreeze all, but protect VGG)
# for param in model.parameters():
#     param.requires_grad = True

# optimizer = optim.AdamW([
#     {'params': model.image_model.backbone.parameters(), 'lr': 1e-6}, # Slow LR for VGG
#     {'params': model.morphology_model.parameters(), 'lr': 1e-4},     # Fast LR for MLP
#     {'params': model.classifier.parameters(), 'lr': 1e-4}            # Fast LR for Classifier
# ], weight_decay=1e-3) # Higher weight decay to prevent overfitting on small data

# scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# # ==========================================
# # 5. THE TRAINING LOOP
# # ==========================================
# NUM_EPOCHS = 40
# SAVE_DIR = r"/content/drive/MyDrive/Major_Project/checkpoints_fusion_subset"
# os.makedirs(SAVE_DIR, exist_ok=True)

# best_acc = 0.0
# best_model_wts = copy.deepcopy(model.state_dict())
# epochs_no_improve = 0
# patience = 99

# print("\n" + "="*50)
# print(f"🚀 STARTING FINE-TUNING ON CLEAN SUBSET (Batch Size: {BATCH_SIZE})")
# print("="*50)

# for epoch in range(NUM_EPOCHS):
#     model.train()
#     running_loss, running_corrects, processed = 0.0, 0, 0
#     current_lr = optimizer.param_groups[1]['lr'] # Track MLP learning rate

#     train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]", leave=False)

#     for images, vectors, labels in train_pbar:
#         images, vectors, labels = images.to(device), vectors.to(device), labels.to(device)

#         optimizer.zero_grad()
#         outputs = model(image=images, vector=vectors)
#         _, preds = torch.max(outputs, 1)

#         loss = criterion(outputs, labels)
#         loss.backward()
#         torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
#         optimizer.step()

#         batch_sz = images.size(0)
#         running_loss += loss.item() * batch_sz
#         running_corrects += torch.sum(preds == labels.data)
#         processed += batch_sz

#         train_pbar.set_postfix({'loss': f"{loss.item():.4f}", 'acc': f"{(running_corrects.double()/processed).item():.4f}"})

#     train_loss = running_loss / len(train_dataset)
#     train_acc = running_corrects.double() / len(train_dataset)

#     # ---------- Validation Phase ----------
#     model.eval()
#     val_loss, val_corrects = 0.0, 0

#     with torch.no_grad():
#         for images, vectors, labels in val_loader:
#             images, vectors, labels = images.to(device), vectors.to(device), labels.to(device)
#             outputs = model(image=images, vector=vectors)
#             _, preds = torch.max(outputs, 1)
#             loss = criterion(outputs, labels)

#             val_loss += loss.item() * images.size(0)
#             val_corrects += torch.sum(preds == labels.data)

#     val_loss = val_loss / len(val_dataset)
#     val_acc = val_corrects.double() / len(val_dataset)
#     scheduler.step(val_loss)

#     print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | LR: {current_lr:.1e}")

#     # ---------- Checkpoints & Early Stopping ----------
#     if val_acc > best_acc:
#         best_acc = val_acc
#         best_model_wts = copy.deepcopy(model.state_dict())
#         epochs_no_improve = 0
#     else:
#         epochs_no_improve += 1

#     if epochs_no_improve >= patience:
#         print(f"[*] Early stopping activated at Epoch {epoch+1}!")
#         break

# # ---------- Save the Best Model ----------
# model_path = os.path.join(SAVE_DIR, "best_fusion_subset.pth")
# torch.save(best_model_wts, model_path)
# print("\n" + "="*50)
# print(f"🎉 TRAINING COMPLETE. Best Validation Accuracy: {best_acc:.4f}")
# print(f"💾 Model successfully saved to: {model_path}")
# print("="*50)

In [ ]:
class FusionModel(nn.Module):
    def __init__(self, image_model_name: str = "vgg16_bn", image_output_dim: int = 256,
                 mlp_input_dim: int = 27, mlp_hidden_dims: list = [64, 128],
                 mlp_output_dim: int = 256, num_classes: int = 2, pretrained: bool = True):
        super(FusionModel, self).__init__()
        self.image_model = ImageModel(model_name=image_model_name, output_dim=image_output_dim, pretrained=pretrained)
        self.morphology_model = MorphologyModel(input_dim=mlp_input_dim, hidden_dims=mlp_hidden_dims, output_dim=mlp_output_dim)

        fused_dim = image_output_dim + mlp_output_dim

        self.classifier = nn.Sequential(
            nn.LayerNorm(fused_dim),
            # Giảm nơ-ron từ 128 xuống 64 để chống học vẹt
            nn.Linear(fused_dim, 64),
            nn.ReLU(),
            # Tăng Dropout lên 0.6 (Xóa ngẫu nhiên 60% kết nối mỗi lần học)
            nn.Dropout(0.6),
            nn.Linear(64, num_classes)
        )

    def forward(self, image, vector):
        img_feat = self.image_model(image)
        vec_feat = self.morphology_model(vector)
        fused = torch.cat([img_feat, vec_feat], dim=1)
        return self.classifier(fused)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from collections import Counter
from tqdm.auto import tqdm
import copy
import os

# ==========================================
# 2. FEATURE SELECTION & CLEANING
# ==========================================
print("\n🧹 Cleaning data and extracting features...")

# Retrieve the synchronized image paths and labels
synced_image_paths = df_merged['full_image_path'].tolist()
synced_labels = df_merged['label_x'].tolist() if 'label_x' in df_merged.columns else df_merged['label'].tolist()

numeric_df = df_merged.select_dtypes(include=['number', 'float', 'int'])

# Strictly drop metadata to prevent data leakage (including the new 'egg_id')
columns_to_drop = ['O', 'egg_id', 'egg_id_x', 'egg_id_y', 'batch', 'day', 'ngay', 'dot',
                   'side_x', 'side_y', 'img_key', 'img_path', 'image_path',
                   'full_image_path', 'filename', 'label', 'label_x', 'label_y',
                   'Label', 'QA_Failed', 'length_mm', 'width_mm']

numeric_df = numeric_df.drop(columns=[col for col in columns_to_drop if col in numeric_df.columns])

# Handle NaNs and Infs safely
numeric_df = numeric_df.replace([np.inf, -np.inf], np.nan)
numeric_df = numeric_df.fillna(0)

raw_morph_features = numeric_df.values
mlp_input_dim = raw_morph_features.shape[1]
print(f"[*] MLP Input Dimension optimally reduced to: {mlp_input_dim} features.")

# ==========================================
# 3. SPLIT, SCALE, AND DATALOADERS
# ==========================================
# Split the small dataset (80% Train, 20% Val)
indices = np.arange(len(synced_image_paths))
train_idx, val_idx = train_test_split(indices, test_size=0.2, stratify=synced_labels, random_state=42)

train_data_list = [(synced_image_paths[i], synced_labels[i]) for i in train_idx]
val_data_list = [(synced_image_paths[i], synced_labels[i]) for i in val_idx]
train_labels = [synced_labels[i] for i in train_idx]

# Scale features based ONLY on training set
scaler = StandardScaler()
train_scaled_morph = scaler.fit_transform(raw_morph_features[train_idx])
val_scaled_morph = scaler.transform(raw_morph_features[val_idx])

# Using Batch Size 8 since the dataset is very small
BATCH_SIZE = 8

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Assuming FusionEggDataset is already defined in your environment
train_dataset = FusionEggDataset(train_data_list, train_scaled_morph, train_transforms)
val_dataset = FusionEggDataset(val_data_list, val_scaled_morph, val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# ==========================================
# 4. INITIALIZE MODEL & SETUP DIFFERENTIAL LR
# ==========================================
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

model = FusionModel(
    image_model_name="vgg16_bn",
    mlp_input_dim=mlp_input_dim,
    pretrained=False
).to(device)

# Load the custom blood vessel weights into the VGG backbone
weights_path = "/content/drive/MyDrive/Major_Project/checkpoints_bv/vgg16_bn_phase2_bv_fold_2.pth"
if os.path.exists(weights_path):
    state_dict = torch.load(weights_path, map_location=device)
    filtered_state_dict = {k: v for k, v in state_dict.items() if not k.startswith('classifier.6')}
    model.image_model.backbone.load_state_dict(filtered_state_dict, strict=False)
    print(f"[+] Loaded custom VGG weights from {weights_path}")

# Calculate Class Weights
label_counts = Counter(train_labels)
weight_0 = len(train_labels) / (2.0 * label_counts.get(0, 1))
weight_1 = len(train_labels) / (2.0 * label_counts.get(1, 1))
class_weights = torch.tensor([weight_0, weight_1], dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

# # Differential Learning Rates (Unfreeze all, but protect VGG)
# for param in model.parameters():
#     param.requires_grad = True

# optimizer = optim.AdamW([
#     {'params': model.image_model.backbone.parameters(), 'lr': 1e-6}, # Slow LR for VGG
#     {'params': model.morphology_model.parameters(), 'lr': 1e-4},     # Fast LR for MLP
#     {'params': model.classifier.parameters(), 'lr': 1e-4}            # Fast LR for Classifier
# ], weight_decay=1e-3) # Higher weight decay to prevent overfitting on small data

# scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# ---------------------------------------------------------
# 4. OPTIMIZER: STRICT REGULARIZATION FOR SMALL DATA
# ---------------------------------------------------------
print("\n❄️ Absolutely freezing VGG to prevent overfitting on small data...")

# 1. Khóa chặt (Freeze) toàn bộ nhánh VGG
for param in model.image_model.backbone.parameters():
    param.requires_grad = False

# Mở khóa (Unfreeze) nhánh MLP và Classifier
for param in model.morphology_model.parameters():
    param.requires_grad = True
for param in model.classifier.parameters():
    param.requires_grad = True

# 2. Bộ tối ưu hóa: Tăng mạnh Weight Decay (1e-2) để siết chặt kỷ luật
trainable_params = filter(lambda p: p.requires_grad, model.parameters())
optimizer = optim.AdamW(trainable_params, lr=1e-4, weight_decay=1e-2)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4)

# ==========================================
# 5. THE TRAINING LOOP
# ==========================================
NUM_EPOCHS = 40
SAVE_DIR = r"/content/drive/MyDrive/Major_Project/checkpoints_fusion_subset"
os.makedirs(SAVE_DIR, exist_ok=True)

best_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())
epochs_no_improve = 0
patience = 99

print("\n" + "="*50)
print(f"🚀 STARTING FINE-TUNING ON CLEAN SUBSET (Batch Size: {BATCH_SIZE})")
print("="*50)

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss, running_corrects, processed = 0.0, 0, 0
    current_lr = optimizer.param_groups[]['lr'] # Track MLP learning rate

    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]", leave=False)

    for images, vectors, labels in train_pbar:
        images, vectors, labels = images.to(device), vectors.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(image=images, vector=vectors)
        _, preds = torch.max(outputs, 1)

        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()

        batch_sz = images.size(0)
        running_loss += loss.item() * batch_sz
        running_corrects += torch.sum(preds == labels.data)
        processed += batch_sz

        train_pbar.set_postfix({'loss': f"{loss.item():.4f}", 'acc': f"{(running_corrects.double()/processed).item():.4f}"})

    train_loss = running_loss / len(train_dataset)
    train_acc = running_corrects.double() / len(train_dataset)

    # ---------- Validation Phase ----------
    model.eval()
    val_loss, val_corrects = 0.0, 0

    with torch.no_grad():
        for images, vectors, labels in val_loader:
            images, vectors, labels = images.to(device), vectors.to(device), labels.to(device)
            outputs = model(image=images, vector=vectors)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            val_corrects += torch.sum(preds == labels.data)

    val_loss = val_loss / len(val_dataset)
    val_acc = val_corrects.double() / len(val_dataset)
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | LR: {current_lr:.1e}")

    # ---------- Checkpoints & Early Stopping ----------
    if val_acc > best_acc:
        best_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= patience:
        print(f"[*] Early stopping activated at Epoch {epoch+1}!")
        break

# ---------- Save the Best Model ----------
model_path = os.path.join(SAVE_DIR, "best_fusion_subset.pth")
torch.save(best_model_wts, model_path)
print("\n" + "="*50)
print(f"🎉 TRAINING COMPLETE. Best Validation Accuracy: {best_acc:.4f}")
print(f"💾 Model successfully saved to: {model_path}")
print("="*50)


🧹 Cleaning data and extracting features...
[*] MLP Input Dimension optimally reduced to: 27 features.
[+] Loaded custom VGG weights from /content/drive/MyDrive/Major_Project/checkpoints_bv/vgg16_bn_phase2_bv_fold_2.pth

❄️ Absolutely freezing VGG to prevent overfitting on small data...

🚀 STARTING FINE-TUNING ON CLEAN SUBSET (Batch Size: 8)


Epoch 1/40 [Train]:   0%|          | 0/19 [00:00<?, ?it/s]

Epoch 01/40 | Train Loss: 0.7378 Acc: 0.5608 | Val Loss: 0.7033 Acc: 0.6053 | LR: 1.0e-04


Epoch 2/40 [Train]:   0%|          | 0/19 [00:00<?, ?it/s]

Epoch 02/40 | Train Loss: 0.7011 Acc: 0.5203 | Val Loss: 0.7045 Acc: 0.5263 | LR: 1.0e-04


...

Epoch 38/40 | Train Loss: 0.7018 Acc: 0.5608 | Val Loss: 0.7069 Acc: 0.5263 | LR: 1.6e-06


Epoch 39/40 [Train]:   0%|          | 0/19 [00:00<?, ?it/s]

Epoch 39/40 | Train Loss: 0.7003 Acc: 0.5743 | Val Loss: 0.7089 Acc: 0.5263 | LR: 1.6e-06


Epoch 40/40 [Train]:   0%|          | 0/19 [00:00<?, ?it/s]

Epoch 40/40 | Train Loss: 0.6470 Acc: 0.6757 | Val Loss: 0.7130 Acc: 0.5789 | LR: 7.8e-07

🎉 TRAINING COMPLETE. Best Validation Accuracy: 0.6316
💾 Model successfully saved to: /content/drive/MyDrive/Major_Project/checkpoints_fusion_subset/best_fusion_subset.pth
